# Oil Basket Consolidated Index (Colab)

This notebook builds a **free** data pipeline using Yahoo Finance (`yfinance`) for your listed crude contracts.

It does all of the following:
- Downloads up to the last **60 days** of history for each contract (best available interval from Yahoo).
- Every **15 minutes** (or manually), fetches the most recent **20 minutes** of intraday data with `prepost=True` and deduplicates it.
- Creates an **equal-weight consolidated index** and reports changes over:
  - 1 week
  - 2 days
  - 12 hours
  - 6 hours
  - 2 hours
  - 1 hour
  - 30 minutes
  - 15 minutes
  - 5 minutes
- If a market is closed, the latest known value is carried forward (last available point as-of the target timestamp).

> Note: Some requested contracts may not be exposed by Yahoo Finance under a stable ticker. This notebook uses candidate symbol mapping + auto-validation and continues with available ones.

In [ ]:
# @title 1) Install dependencies (Colab)
!pip -q install yfinance pandas pytz

In [ ]:
# @title 2) Imports and configuration
import os
import time
from datetime import datetime, timedelta, timezone

import numpy as np
import pandas as pd
import yfinance as yf

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 50)

# Storage location (works both in Colab and local).
BASE_DIR = '/content/oil_index_data' if os.path.exists('/content') else './oil_index_data'
os.makedirs(BASE_DIR, exist_ok=True)

HISTORY_DIR = os.path.join(BASE_DIR, 'history_60d')
os.makedirs(HISTORY_DIR, exist_ok=True)

INTRADAY_FILE = os.path.join(BASE_DIR, 'intraday_1m_last20m_runs.csv')
METADATA_FILE = os.path.join(BASE_DIR, 'resolved_symbols.csv')

# Contracts and Yahoo candidate tickers.
# You can add/edit candidates over time if Yahoo listing changes.
CONTRACT_CANDIDATES = {
    'CL — NYMEX WTI (Light Sweet Crude Oil)': ['CL=F'],
    'B — ICE Brent Crude': ['BZ=F'],
    'BZ — NYMEX Brent Last Day Financial': ['BZ=F'],
    'DBI — ICE Dubai 1st Line': ['DBI=F', 'DBI.F'],
    'DCB — NYMEX Dubai Crude Oil (Platts) Calendar Swap': ['DCB=F', 'DCB.F'],
    'OQD — DME Oman Crude Oil Futures': ['OQD=F', 'OQD.F'],
    'ADM — ICE Futures Abu Dhabi Murban Crude Oil Futures': ['ADM=F', 'ADM.F'],
    'HOU — ICE Midland WTI (Houston)': ['HOU=F', 'HOU.F'],
    'TMW — ICE Western Canadian Select (WCS) 1A Index': ['TMW=F', 'TMW.F'],
    'ARM — ICE Argus Mars': ['ARM=F', 'ARM.F'],
}

# Time horizons requested
HORIZONS = {
    '1w': pd.Timedelta(days=7),
    '2d': pd.Timedelta(days=2),
    '12h': pd.Timedelta(hours=12),
    '6h': pd.Timedelta(hours=6),
    '2h': pd.Timedelta(hours=2),
    '1h': pd.Timedelta(hours=1),
    '30m': pd.Timedelta(minutes=30),
    '15m': pd.Timedelta(minutes=15),
    '5m': pd.Timedelta(minutes=5),
}

print('BASE_DIR:', BASE_DIR)

In [ ]:
# @title 3) Core helpers
def _standardize_ohlcv(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame()
    df = df.copy()
    # yfinance may return multi-index columns in some cases
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]
    cols = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
    keep = [c for c in cols if c in df.columns]
    df = df[keep].copy()
    idx = pd.to_datetime(df.index, utc=True, errors='coerce')
    df.index = idx
    df = df[~df.index.isna()]
    df = df.sort_index()
    # Prefer Adj Close when available
    if 'Adj Close' in df.columns:
        df['Px'] = df['Adj Close']
    else:
        df['Px'] = df['Close']
    return df

def resolve_symbol(contract_name: str, candidates: list[str]) -> tuple[str | None, pd.DataFrame]:
    """Pick first candidate that returns valid recent data."""
    for sym in candidates:
        try:
            test = yf.download(
                sym,
                period='5d',
                interval='1h',
                auto_adjust=False,
                progress=False,
                prepost=True,
                threads=False,
            )
            test = _standardize_ohlcv(test)
            if not test.empty and test['Px'].notna().sum() > 0:
                return sym, test
        except Exception:
            pass
    return None, pd.DataFrame()

def resolve_all_symbols(contract_candidates: dict) -> pd.DataFrame:
    rows = []
    for contract_name, candidates in contract_candidates.items():
        sym, sample = resolve_symbol(contract_name, candidates)
        rows.append({
            'contract_name': contract_name,
            'resolved_symbol': sym,
            'available': sym is not None,
            'sample_points': 0 if sample.empty else int(sample['Px'].notna().sum()),
        })
    out = pd.DataFrame(rows).sort_values(['available', 'contract_name'], ascending=[False, True])
    out.to_csv(METADATA_FILE, index=False)
    return out

def download_60d_history(symbol: str) -> pd.DataFrame:
    # 60 days at 30m interval gives enough granularity and full range on Yahoo.
    hist = yf.download(
        symbol,
        period='60d',
        interval='30m',
        auto_adjust=False,
        progress=False,
        prepost=True,
        threads=False,
    )
    return _standardize_ohlcv(hist)

def fetch_last_20m_intraday(symbol: str) -> pd.DataFrame:
    intr = yf.download(
        symbol,
        period='1d',
        interval='1m',
        auto_adjust=False,
        progress=False,
        prepost=True,
        threads=False,
    )
    intr = _standardize_ohlcv(intr)
    if intr.empty:
        return intr
    cutoff = pd.Timestamp.now(tz='UTC') - pd.Timedelta(minutes=20)
    intr = intr[intr.index >= cutoff].copy()
    return intr

def append_intraday_dedup(new_rows: pd.DataFrame) -> pd.DataFrame:
    expected_cols = ['timestamp_utc', 'contract_name', 'symbol', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'Px']
    if new_rows.empty:
        if os.path.exists(INTRADAY_FILE):
            return pd.read_csv(INTRADAY_FILE, parse_dates=['timestamp_utc'])
        return pd.DataFrame(columns=expected_cols)

    df_new = new_rows.copy()
    for c in expected_cols:
        if c not in df_new.columns:
            df_new[c] = np.nan

    if os.path.exists(INTRADAY_FILE):
        old = pd.read_csv(INTRADAY_FILE, parse_dates=['timestamp_utc'])
    else:
        old = pd.DataFrame(columns=expected_cols)

    combined = pd.concat([old, df_new[expected_cols]], ignore_index=True)
    combined['timestamp_utc'] = pd.to_datetime(combined['timestamp_utc'], utc=True, errors='coerce')
    combined = combined.dropna(subset=['timestamp_utc', 'symbol'])
    combined = combined.sort_values('timestamp_utc')
    combined = combined.drop_duplicates(subset=['timestamp_utc', 'symbol'], keep='last')
    combined.to_csv(INTRADAY_FILE, index=False)
    return combined

def build_symbol_price_series(history_df: pd.DataFrame, intraday_all: pd.DataFrame, symbol: str) -> pd.Series:
    parts = []
    if history_df is not None and not history_df.empty:
        s1 = history_df['Px'].dropna().copy()
        s1.index = pd.to_datetime(s1.index, utc=True)
        parts.append(s1)
    if intraday_all is not None and not intraday_all.empty:
        x = intraday_all[intraday_all['symbol'] == symbol].copy()
        if not x.empty:
            x['timestamp_utc'] = pd.to_datetime(x['timestamp_utc'], utc=True, errors='coerce')
            x = x.dropna(subset=['timestamp_utc', 'Px'])
            s2 = x.set_index('timestamp_utc')['Px'].astype(float).sort_index()
            parts.append(s2)

    if not parts:
        return pd.Series(dtype=float)

    s = pd.concat(parts).sort_index()
    s = s[~s.index.duplicated(keep='last')]
    return s

def asof_value(series: pd.Series, t: pd.Timestamp) -> float:
    if series.empty:
        return np.nan
    sub = series.loc[:t]
    if sub.empty:
        return np.nan
    return float(sub.iloc[-1])

def compute_equal_weight_returns(price_map: dict[str, pd.Series], horizons: dict[str, pd.Timedelta]) -> pd.DataFrame:
    now = pd.Timestamp.now(tz='UTC')
    out_rows = []

    for label, delta in horizons.items():
        t0 = now - delta
        instrument_returns = []
        contributors = 0

        for sym, s in price_map.items():
            p_now = asof_value(s, now)
            p_then = asof_value(s, t0)
            if np.isfinite(p_now) and np.isfinite(p_then) and p_then != 0:
                r = (p_now / p_then) - 1.0
                instrument_returns.append(r)
                contributors += 1

        ew_ret = float(np.mean(instrument_returns)) if instrument_returns else np.nan
        out_rows.append({
            'horizon': label,
            'asof_utc': now,
            'from_utc': t0,
            'equal_weight_return': ew_ret,
            'equal_weight_return_pct': ew_ret * 100 if np.isfinite(ew_ret) else np.nan,
            'contributors': contributors,
        })

    return pd.DataFrame(out_rows).sort_values('from_utc')

In [ ]:
# @title 4) Resolve symbols and download 60-day history
resolved = resolve_all_symbols(CONTRACT_CANDIDATES)
display(resolved)

available = resolved[resolved['available']].copy()
if available.empty:
    raise RuntimeError('No symbols could be resolved from Yahoo candidates. Edit CONTRACT_CANDIDATES and retry.')

history_cache = {}
for _, row in available.iterrows():
    contract_name = row['contract_name']
    symbol = row['resolved_symbol']
    hist = download_60d_history(symbol)
    history_cache[symbol] = hist

    out_file = os.path.join(HISTORY_DIR, f"{symbol.replace('=','_')}_60d_30m.csv")
    h = hist.copy()
    h = h.reset_index().rename(columns={'index':'timestamp_utc'})
    h.to_csv(out_file, index=False)
    print(f'Saved 60d history: {contract_name} | {symbol} | rows={len(hist)} -> {out_file}')

In [ ]:
# @title 5) Single collection run (manual trigger)
def run_collection_once(resolved_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in resolved_df[resolved_df['available']].iterrows():
        contract_name = row['contract_name']
        symbol = row['resolved_symbol']
        try:
            intr = fetch_last_20m_intraday(symbol)
            if intr.empty:
                continue
            tmp = intr.reset_index().rename(columns={'index': 'timestamp_utc'})
            tmp['contract_name'] = contract_name
            tmp['symbol'] = symbol
            rows.append(tmp)
        except Exception as e:
            print(f'[WARN] Failed intraday fetch for {symbol}: {e}')

    if rows:
        new_rows = pd.concat(rows, ignore_index=True)
    else:
        new_rows = pd.DataFrame()

    consolidated_intraday = append_intraday_dedup(new_rows)
    print(f'Intraday consolidated rows in storage: {len(consolidated_intraday)}')
    return consolidated_intraday

intraday_all = run_collection_once(resolved)
display(intraday_all.tail(20))

In [ ]:
# @title 6) Optional scheduler (every 15 minutes)
# Set RUN_SCHEDULER=True to keep collecting automatically in the notebook session.
RUN_SCHEDULER = False
MAX_RUNS = 8  # safety cap; adjust as needed

if RUN_SCHEDULER:
    for i in range(MAX_RUNS):
        print(f'\n=== Scheduled run {i+1}/{MAX_RUNS} @ {datetime.now(timezone.utc).isoformat()} ===')
        _ = run_collection_once(resolved)
        if i < MAX_RUNS - 1:
            time.sleep(15 * 60)

In [ ]:
# @title 7) Build equal-weight consolidated index returns for requested horizons
# Reload intraday storage (if exists)
if os.path.exists(INTRADAY_FILE):
    intraday_all = pd.read_csv(INTRADAY_FILE, parse_dates=['timestamp_utc'])
else:
    intraday_all = pd.DataFrame(columns=['timestamp_utc', 'symbol', 'Px'])

# Build price series per available symbol by combining 60d history + deduped intraday
price_map = {}
for _, row in resolved[resolved['available']].iterrows():
    symbol = row['resolved_symbol']
    hist = history_cache.get(symbol)
    if hist is None:
        # fallback from file if notebook restarted
        f = os.path.join(HISTORY_DIR, f"{symbol.replace('=','_')}_60d_30m.csv")
        if os.path.exists(f):
            h = pd.read_csv(f, parse_dates=['timestamp_utc'])
            h = h.set_index(pd.to_datetime(h['timestamp_utc'], utc=True))
            hist = h
        else:
            hist = pd.DataFrame()

    s = build_symbol_price_series(hist, intraday_all, symbol)
    if not s.empty:
        price_map[symbol] = s

index_returns = compute_equal_weight_returns(price_map, HORIZONS)
display(index_returns[['horizon', 'equal_weight_return_pct', 'contributors', 'from_utc', 'asof_utc']].sort_values('contributors', ascending=False))

In [ ]:
# @title 8) Optional: show per-instrument returns table for transparency
now = pd.Timestamp.now(tz='UTC')
rows = []
for symbol, s in price_map.items():
    r = {'symbol': symbol}
    p_now = asof_value(s, now)
    r['px_now'] = p_now
    for label, delta in HORIZONS.items():
        p_then = asof_value(s, now - delta)
        if np.isfinite(p_now) and np.isfinite(p_then) and p_then != 0:
            rr = (p_now / p_then - 1.0) * 100
        else:
            rr = np.nan
        r[f'{label}_pct'] = rr
    rows.append(r)

per_instrument = pd.DataFrame(rows).sort_values('symbol')
display(per_instrument)

## Notes
- For contracts not available from Yahoo, their rows remain unresolved and are excluded from the equal-weight average until you provide a working symbol source.
- Equal-weighting is done across **currently contributing instruments** per horizon (those with both `now` and `then` as-of prices).
- Carry-forward behavior is implemented with as-of lookup (latest value at or before the target timestamp), which naturally handles exchange closures and non-overlapping sessions.